In [6]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [7]:
# Start with our previous MLP
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
s_to_i = {s: i+1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
i_to_s = {i:s for s, i in s_to_i.items()}

In [8]:
block_size = 3

# create the dataset, now we have 228146 datapoints
# implement test and training splits
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w+'.':
            ix = s_to_i[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# create the datasets
X_train, Y_train = build_dataset(words[:n1])
X_val, Y_val = build_dataset(words[n1:n2])
X_test, Y_test = build_dataset(words[n2:])

In [9]:
# set up net

num_chars = 27
emb_dim = 10
hidden_layer_size = 200

# initialise
g = torch.Generator().manual_seed(2147483647)

# embedding matrix
C = torch.randn((num_chars, emb_dim), generator=g)

# one linear layer, one batch norm layer, tanh layer, then output

W1 = torch.randn((emb_dim * block_size, hidden_layer_size), generator=g) / (emb_dim*num_chars)**0.5 # may as well scale it
# no need for b1 since we batch normin

W2 = torch.randn((hidden_layer_size, num_chars), generator=g) / (hidden_layer_size*num_chars)**0.5 # may as well scale it
b2 = torch.randn(num_chars, generator=g) * 0.01 # may as well make it small

# batch norm params - init as ones and zeros so our preactivations start off standard normal
gamma = torch.ones(hidden_layer_size)
beta = torch.zeros(hidden_layer_size)

# running batch norm totals
running_mean = torch.zeros(hidden_layer_size)
running_var = torch.ones(hidden_layer_size)

# full params
params = [C, W1, W2, b2, gamma, beta]
for p in params:
    p.requires_grad = True

In [14]:
# training loop
iterations = 100000
batch_size = 32
momentum = 0.01
eps = 1e-5

for i in range(iterations):

    # batches
    ix = torch.randint(0, X_train.shape[0], (batch_size, ), generator=g)
    Xb, Yb = X_train[ix], Y_train[ix]

    # embeddings
    emb = C[Xb].view(Xb.shape[0], -1)

    # first layer
    pre_act = emb @ W1 # linear

    # batchnorm
    batch_mean = pre_act.mean(0, keepdim=True)
    batch_var = pre_act.var(0, keepdim=True)

    pre_act_norm = (pre_act - batch_mean) / torch.sqrt(batch_var + eps)
    pre_act_scaled = gamma * pre_act_norm + beta

    act = torch.tanh(pre_act_scaled) # tanh

    # output layer
    logits = act @ W2 + b2

    # loss
    loss = F.cross_entropy(logits, Yb)

    # backward pass
    for p in params:
        p.grad = None
    loss.backward()

    # gradient descent
    lr = 0.1 if i < int(iterations/2) else 0.01
    for p in params:
        p.data += -lr * p.grad

    # running mean and var updates
    with torch.no_grad():
        running_mean = momentum * batch_mean + (1-momentum) * running_mean
        running_var = momentum * batch_var + (1-momentum) * running_var


    if i % 10000 == 0:
        print(f"iteration {i}: loss = {loss.item()}")

iteration 0: loss = 2.5970964431762695
iteration 10000: loss = 2.177128553390503
iteration 20000: loss = 1.8926808834075928
iteration 30000: loss = 2.1780192852020264
iteration 40000: loss = 2.0670530796051025
iteration 50000: loss = 2.1657180786132812
iteration 60000: loss = 2.412999153137207
iteration 70000: loss = 1.8497146368026733
iteration 80000: loss = 2.361705780029297
iteration 90000: loss = 1.9006177186965942


In [ ]:
# try some inference on val set
# embeddings
emb = C[X_val].view(X_val.shape[0], -1)

# first layer
pre_act = emb @ W1 # linear

pre_act_norm = (pre_act - running_mean) / torch.sqrt(running_var + eps)
pre_act_scaled = gamma * pre_act_norm + beta

act = torch.tanh(pre_act_scaled) # tanh

# output layer
logits = act @ W2 + b2

# loss
loss = F.cross_entropy(logits, Y_val)

# print(loss)


tensor(2.1319, grad_fn=<NllLossBackward0>)


In [ ]:
# try the folding batch norm into weights method
# embeddings
emb = C[X_val].view(X_val.shape[0], -1)

# change W1 to fold in the batch norm params
scaling_factor = gamma / torch.sqrt(running_var + eps)
W_batch = W1 * scaling_factor

# now we need to add a bias
b_batch = -running_mean * scaling_factor + beta

# now we run our linear pass as normal without batchnorm
pre_act_scaled = emb @ W_batch + b_batch

act = torch.tanh(pre_act_scaled) # tanh

# output layer
logits = act @ W2 + b2

# loss
loss = F.cross_entropy(logits, Y_val)

print(loss) # SAME LOSS AS ABOVE


tensor(2.1319, grad_fn=<NllLossBackward0>)
